# Dynamic Programming

DP is an optimization technique for problems with:

1. **Optimal substructure** -- optimal solution can be built from optimal solutions of subproblems
2. **Overlapping subproblems** -- same subproblems are solved repeatedly

## Two Approaches

| Approach | Direction | Technique |
|----------|-----------|----------|
| Top-down | Start from original problem, recurse down | Memoization (cache results) |
| Bottom-up | Start from smallest subproblems, build up | Tabulation (fill a table) |

Both avoid recomputing subproblems. Bottom-up avoids recursion overhead.

# Fibonacci

Classic example to illustrate the problem with naive recursion and how DP fixes it.

```
fib(5)
├── fib(4)
│   ├── fib(3)        ← computed again below
│   │   ├── fib(2)
│   │   └── fib(1)
│   └── fib(2)        ← computed again
└── fib(3)            ← same subtree as above
    ├── fib(2)
    └── fib(1)
```

The tree has O(2ⁿ) nodes but only n *distinct* values in it -- that gap between "calls made"
and "answers that exist" is what DP eliminates. The four versions below are the same
recurrence with progressively less waste:

| Version | Idea | Time | Space |
|---|---|---|---|
| `fib_naive` | recompute everything | O(2ⁿ) | O(n) call stack |
| `fib_memo` | cache each answer the first time it is computed (top-down) | O(n) | O(n) |
| `fib_tab` | fill a table from the base cases upward (bottom-up) | O(n) | O(n) |
| `fib_opt` | keep only the two values the recurrence actually reads | O(n) | O(1) |

`fib_opt` is worth a second look: `dp[i]` only ever depends on `dp[i-1]` and `dp[i-2]`, so
the full table is dead weight and two variables suffice. Recognising that "the recurrence
only looks k rows back" is the standard route from O(n) to O(1) space -- or from O(n×m) to
O(m) for the 2-D problems further down.

In [ ]:
def fib_naive(n):
    """O(2^n) time -- exponential due to overlapping subproblems."""
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)


def fib_memo(n, memo=None):
    """Top-down with memoization. O(n) time, O(n) space."""
    if memo is None:
        memo = {}  # a fresh cache per top-level call, not a shared default
    if n <= 1:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]


def fib_tab(n):
    """Bottom-up tabulation. O(n) time, O(n) space."""
    if n <= 1:
        return n
    dp = [0] * (n + 1)
    dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]


def fib_opt(n):
    """Space-optimized -- the recurrence only looks two steps back. O(1) space."""
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b


def test_fib():
    expected = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
    for i, val in enumerate(expected):
        assert fib_naive(i) == val
        assert fib_memo(i) == val
        assert fib_tab(i) == val
        assert fib_opt(i) == val
    # the memo version scales where the naive one cannot
    assert fib_memo(90) == 2880067194370816120
    assert fib_opt(90) == fib_memo(90)


test_fib()

# Coin Change (Minimum Coins)

Given coin denominations and a target amount, find the **minimum number of coins** to make
that amount.

**Why not greedy?** With `coins = [1, 5, 6, 9]` and `amount = 11`, taking the largest coin
first gives 9 + 1 + 1 = three coins. The optimum is 6 + 5 = **two**. Greedy fails because
choosing a coin changes which combinations remain reachable, so every option has to be
explored -- which is exactly what the recurrence does.

**Recurrence:** `dp[i] = min(dp[i - coin] + 1)` over every coin that fits in `i`.

Read it as: "to make i, take some coin, then make the rest optimally". `dp[0] = 0` is the
base case (nothing needed for nothing), and `inf` marks amounts no combination can reach.

```
coins = [1, 5, 6, 9]

amount   0  1  2  3  4  5  6  7  8  9 10 11
dp       0  1  2  3  4  1  1  2  3  1  2  2
                                          ↑
dp[11] = min( dp[10] + 1,     using a 1  → 3
              dp[6]  + 1,     using a 5  → 2   ← best
              dp[5]  + 1,     using a 6  → 2
              dp[2]  + 1 )    using a 9  → 3
```

Notice `dp[9] = 1` -- the 9 coin -- yet the answer for 11 routes through `dp[6]` instead.
Optimal sub-answers do not have to build on the *largest* coin, only on some coin.

**Time:** O(amount × len(coins)) &nbsp; **Space:** O(amount)

In [ ]:
import math

def coin_change(coins, amount):
    """Bottom-up tabulation."""
    dp = [math.inf] * (amount + 1)
    dp[0] = 0  # 0 coins needed for amount 0
    for i in range(1, amount + 1):
        for coin in coins:
            if coin <= i and dp[i - coin] + 1 < dp[i]:
                dp[i] = dp[i - coin] + 1
    return dp[amount] if dp[amount] != math.inf else -1

def test_coin_change():
    assert coin_change([1, 5, 6, 9], 11) == 2   # 6+5
    assert coin_change([1, 5, 10, 25], 30) == 2  # 25+5
    assert coin_change([2], 3) == -1              # impossible
    assert coin_change([1], 0) == 0

test_coin_change()

# Longest Common Subsequence (LCS)

Given two strings, find the length of their longest common **subsequence** -- characters in
order, but not necessarily adjacent.

**Example:** `"ABCBDAB"` and `"BDCAB"` → LCS is `"BCAB"`, length 4

Compare the last characters of the two prefixes. There are only two situations:

- **They match.** That character can safely be part of the LCS, so the answer is 1 plus the
  LCS of both prefixes with that character removed → `dp[i-1][j-1] + 1`
- **They differ.** At least one of the two characters is not in the LCS, so try dropping
  each and keep the better outcome → `max(dp[i-1][j], dp[i][j-1])`

`dp[i][j]` is the LCS length of the first i characters of `s1` and the first j of `s2`. The
table is `(m+1) × (n+1)` so that row 0 and column 0 can hold the base case: an empty string
shares nothing.

```
lcs("ABC", "AC")

            ""   A   C
      ""     0   0   0
      A      0   1   1      A == A → dp[0][0] + 1 = 1
      B      0   1   1      B vs A, B vs C → carry the best neighbour
      C      0   1   2      C == C → dp[2][1] + 1 = 2

answer: dp[3][2] = 2   ("AC")
```

Each cell reads only the row above and the cell to the left, which is why the table can be
filled in a single pass -- and why O(n) space is possible by keeping just one row.

**Time:** O(m × n) &nbsp; **Space:** O(m × n)

In [ ]:
def lcs(s1, s2):
    """Bottom-up tabulation."""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]

def test_lcs():
    assert lcs('ABCBDAB', 'BDCAB') == 4  # BCAB
    assert lcs('ABC', 'AC') == 2          # AC
    assert lcs('ABC', 'DEF') == 0
    assert lcs('', 'ABC') == 0

test_lcs()

# 0/1 Knapsack

Given items with weights and values, and a capacity, find the **maximum value** that fits.
Each item may be taken at most once -- hence "0/1".

For each item there are only two choices, so the recurrence is a two-way max:

- **Skip it:** the value is whatever the previous items achieved at this capacity →
  `dp[i-1][w]`
- **Take it** (only if it fits): its value plus the best achievable with the *remaining*
  capacity and the *previous* items → `val[i-1] + dp[i-1][w - wt[i-1]]`

`dp[i][w]` = best value using the first i items within capacity w. Every read is from row
`i-1`, which is precisely what enforces "at most once" -- an item can never be counted again
inside its own row. (Allow `dp[i][w - wt]` instead and you have the *unbounded* knapsack,
where items may repeat.)

```
wt  = [1, 3, 4, 5]     val = [1, 4, 5, 7]     capacity 7

                w=0  1  2  3  4  5  6  7
  no items       0   0  0  0  0  0  0  0
  + item 1       0   1  1  1  1  1  1  1
  + item 2       0   1  1  4  5  5  5  5
  + item 3       0   1  1  4  5  6  6  9   ← 4 + 5, weights 3 + 4 = 7
  + item 4       0   1  1  4  5  7  8  9

answer: dp[4][7] = 9   (items 2 and 3)
```

Item 4 is the most valuable single item (7) and still is not part of the answer -- the pair
2+3 fills the capacity better. That is the greedy trap again, and the reason for the table.

**Time:** O(n × W) &nbsp; **Space:** O(n × W), reducible to O(W) with a single row

In [ ]:
def knapsack(wt, val, capacity):
    """Bottom-up tabulation."""
    n = len(wt)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for w in range(1, capacity + 1):
            dp[i][w] = dp[i - 1][w]  # don't take item i
            if wt[i - 1] <= w:
                dp[i][w] = max(dp[i][w], val[i - 1] + dp[i - 1][w - wt[i - 1]])
    return dp[n][capacity]

def test_knapsack():
    assert knapsack([1, 3, 4, 5], [1, 4, 5, 7], 7) == 9  # items 2+3 (4+5 val, 3+4 wt)
    assert knapsack([2, 3, 4], [3, 4, 5], 5) == 7        # items 1+2
    assert knapsack([10], [100], 5) == 0                  # item too heavy

test_knapsack()

# Python Built-in: `functools.lru_cache`

Python provides automatic memoization via `@lru_cache` decorator.
This turns any recursive function into a top-down DP solution with one line.

In [ ]:
from functools import lru_cache

# naive recursive fib becomes O(n) with one decorator
@lru_cache(maxsize=None)
def fib(n):
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(50))  # 12586269025 -- instant, would be impossible without memoization

# cache info shows hits vs misses
print(fib.cache_info())  # CacheInfo(hits=48, misses=51, ...)

# Python 3.9+ also has @cache (unlimited, simpler)
# from functools import cache